# Robust Quad-Based Audio Fingerprinting

## Fingerprint Extraction

In [10]:
import pickle
import os
import matplotlib.pyplot as plt
import numpy as np
from pydub import AudioSegment
from __future__ import division
from scipy.ndimage import maximum_filter, minimum_filter
from bisect import bisect_left
from collections import namedtuple
from heapq import nlargest
from numpy.lib import stride_tricks
import glob
import os
import time

In [11]:
def stft(samples, framesize=1024, hopsize=32):
    """
    Short time fourier transform of audio
    Framesize of 1024 samples (128ms) and
    Hopsize of 32 samples (4ms) as per Sonnleitner/Widmer paper
    Returns: 2D numpy array of float64 values
    """
    window = np.hanning(framesize)
    samples = np.append(np.zeros(int(framesize / 2)), samples)
    cols = int(np.ceil((len(samples) - framesize) / float(hopsize)) + 1)
    samples = np.append(samples, np.zeros(framesize))
    strides = (samples.strides[0] * hopsize, samples.strides[0])
    frames = stride_tricks.as_strided(samples,
                                      shape=(cols, framesize),
                                      strides=strides).copy()
    frames *= window
    spec = np.fft.rfft(frames)
    with np.errstate(divide='ignore'):  # silences "divide by zero" error
        spec = 20. * np.log10(np.abs(spec) / 10e-6)  # amplitude to decibel
    spec[spec == -np.inf] = 0
    return spec


def find_peaks(spec, maxWidth, maxHeight, minWidth=3, minHeight=3):
    """
    Calculate peaks of spectrogram using maximum filter
    Local minima used to filter out uniform areas (e.g. silence)
    Returns: list of namedtuple Peaks
    """
    Peak = namedtuple('Peak', ['x', 'y'])
    maxFilterDimen = (maxWidth, maxHeight)
    minFilterDimen = (minWidth, minHeight)
    maxima = maximum_filter(spec, footprint=np.ones(
        maxFilterDimen, dtype=np.int8))
    minima = minimum_filter(spec, footprint=np.ones(
        minFilterDimen, dtype=np.int8))
    peaks = ((spec == maxima) == (maxima != minima))
    # todo: parabolic interpolation
    x, y = np.nonzero(peaks)
    namedpeaks = [Peak(p[0], p[1]) for p in zip(x, y)]
    return namedpeaks


def n_strongest(spec, quads, n):
    """
    Returns list of n strongest quads in each 1 second partition
    Strongest is calculated by magnitudes of C and D in quad
    """
    strongest = []
    partitions = _find_partitions(quads)
    key = lambda p: (spec[p.C.x][p.C.y] + spec[p.D.x][p.D.y])
    for i in range(1, len(partitions)):
        start = partitions[i - 1]
        end = partitions[i]
        strongest += nlargest(n, quads[start:end], key)
    return strongest


def _find_partitions(quads, l=250):
    """
    Returns list of indices where partitions of 250 (1 second) are
    """
    b_l = bisect_left
    last_x = quads[-1].A.x
    num_partitions = last_x // l
    # creates a tuple of same form as the Quad namedtuple for bisecting
    q = lambda x: ((x,), (), (), ())
    partitions = [b_l(quads, q(i * l)) for i in range(num_partitions)]
    partitions.append(len(quads))
    return partitions


def generate_hash(quad):
    """
    Compute translation- and scale-invariant hash from a given quad
    """
    A, C, D, B = quad
    B = (B.x - A.x, B.y - A.y)
    C = (C.x - A.x, C.y - A.y)
    D = (D.x - A.x, D.y - A.y)
    cDash = (C[0] / B[0], C[1] / B[1])
    dDash = (D[0] / B[0], D[1] / B[1])
    return cDash + dDash

In [12]:
def load_audio(path, downsample=True, normalize=False, target_dBFS=-20.0, snip=None):
    """
    Creates array of samples from input audio file
    snip = only return first n seconds of input
    """
    audio = AudioSegment.from_file(path)
    if downsample:
        # if stereo, sample rate > 8kHz, or > 16-bit depth
        if (audio.channels > 1) \
                or (audio.frame_rate != 8000) \
                or (audio.sample_width != 2):
            audio = _downsample(audio)
    if normalize and audio.dBFS is not target_dBFS:
        audio = _normalize(audio, target_dBFS)
    """if snip > audio.duration_seconds:
        raise InvalidAudioLength(
            "Provided snip length is longer than audio file")"""
    if snip is not None:
        milliseconds = snip * 1000
        audio = audio[:milliseconds]
    return audio.get_array_of_samples()


def _downsample(audio, numChannels=1, sampleRate=8000, bitDepth=2):
    """
    Returns downsampled AudioSegment
    """
    audio = audio.set_channels(numChannels)
    audio = audio.set_frame_rate(sampleRate)
    audio = audio.set_sample_width(bitDepth)
    return audio


def _normalize(audio, target_dBFS):
    """
    Normalizes loudness of AudioSegment
    """
    change_in_dBFS = target_dBFS - audio.dBFS
    return audio.apply_gain(change_in_dBFS)


In [13]:
# from __future__ import division

# from bisect import bisect_left, bisect_right
# from collections import namedtuple

# class SegmentTree:
#     def __init__(self, points):
#         self.points = points
#         self.n = len(points)
#         self.tree = [[] for _ in range(4 * self.n)]
#         self._build(1, 0, self.n - 1)

#     def _build(self, node, l, r):
#         if l == r:
#             p = self.points[l]
#             self.tree[node] = [p]
#         else:
#             mid = (l + r) // 2
#             self._build(2 * node, l, mid)
#             self._build(2 * node + 1, mid + 1, r)
#             self.tree[node] = self.tree[2*node] + self.tree[2*node+1]
#             self.tree[node].sort(key=lambda p: p.y)  # ordena por y para bisect

#     def query(self, node, l, r, ql, qr, Dy_min):
#         if qr < l or ql > r:
#             return []
#         if ql <= l and r <= qr:
#             arr = self.tree[node]
#             from bisect import bisect_left
#             idx = bisect_left([p.y for p in arr], Dy_min)
#             return arr[idx:]
#         mid = (l + r) // 2
#         return self.query(2*node, l, mid, ql, qr, Dy_min) + \
#                self.query(2*node+1, mid+1, r, ql, qr, Dy_min)
    
#     def query_all(self):
#         """Retorna todos os pontos, podendo futuramente aplicar filtro rápido"""
#         return self.points

# def find_quads_seg_tree(peaks, r, c):
#     """
#     Retorna lista de quads válidos a partir dos picos
#     """
#     quads = []
#     for root in peaks:
#         quads += _root_quads_seg_tree(root, peaks, r, c)
#     return quads


# def _root_quads_seg_tree(root, peaks, r, c):
#     """
#     Gera quads para um pico root
#     """
#     filtered = _filter_peaks_seg_tree(root, peaks, r, c)
#     if filtered is None:
#         return []
#     filtered.sort(key=lambda p: p.x)
#     # Construímos a Segment Tree para esse subconjunto
#     segtree = SegmentTree(filtered)

#     found = _valid_quads_seg_tree(root, filtered, segtree)
#     return found if found else []


# def _filter_peaks_seg_tree(root, peaks, r, c):
#     """
#     Seleciona picos na janela de Ax + c ± (r / 2)
#     """
#     lastPeak = peaks[-1].x
#     windowStart = root.x + c - (r / 2)
#     if windowStart > lastPeak:
#         return None
#     windowEnd = windowStart + r
#     idx_start = bisect_left(peaks, (windowStart, 0))
#     idx_end = bisect_right(peaks, (windowEnd, 0))
#     filtered = peaks[idx_start:idx_end]
#     if len(filtered) < 3:
#         return None
#     return filtered


# def _valid_quads_seg_tree(root, filtered, segtree):
#     """
#     Gera quads válidos usando árvore de combinações + segment tree
#     """
#     Quad = namedtuple('Quad', ['A', 'C', 'D', 'B'])
#     validQuads = []

#     # Nível 1: escolha de C
#     for C in filtered:
#         if C == root:
#             continue
#         if not (C.x > root.x and C.y > root.y):
#             continue  # Ay < Cy, Ax < Cx

#         # Nível 2: escolha de D
#         for D in filtered:
#             if D in (root, C):
#                 continue
#             if not (D.x >= C.x):
#                 continue  # Ax < Cx <= Dx

#             # Nível 3: escolha de B via segment tree
#             # Consulta todos os pontos B com Bx >= Dx
#             B_candidates = segtree.query_all()  # todos os pontos
#             for B in B_candidates:
#                 if B in (root, C, D):
#                     continue
#                 if B.x < D.x:
#                     continue  # Dx <= Bx

#                 # Verificação completa das desigualdades
#                 if (root.y < B.y and  # Ay < By
#                     D.y <= B.y and     # Dy <= By
#                     C.y > root.y and   # Ay < Cy (já garantido)
#                     root.x < C.x <= D.x <= B.x):
#                     validQuads.append(Quad(root, C, D, B))

#     return validQuads if validQuads else None

In [14]:
class fpType:
    """
    Parameters for reference/query fingerprint types
    Presented in order (q, r, c, w, h)

    Tuple is used to ensure immutability

    q = quads to create per root point (A)
    r = width of search window
    c = distance from root point to position window
    w = width of max filter
    h = height of max filter

    based on stft hop-size of 32 samples (4ms):
    ref.r =    800ms / 4ms =  200
    ref.c =   1375ms / 4ms = ~345
    que.r =   1300ms / 4ms =  325
    que.c = 1437.5ms / 4ms = ~360

    query filter height/width are calculated as:
    query.w = ref.w / (1 + .2) = 125
    query.h = ref.h * (1 - .2) = 60

    reference width changed from 151 to 150 so that
    result is an int for epsilon of .2 (20% change in speed/tempo)
    """
    #             Q    R    C    W    H
    Reference = (9, 200, 325, 150,  75)
    Query = (500, 345, 360, 125,  60)


class Fingerprint:

    def __init__(self, path, fp_type):
        self.path = path
        if fp_type is not fpType.Reference and fp_type is not fpType.Query:
            raise TypeError(
                "Fingerprint must be of type 'Reference' or 'Query'")
        else:
            self.params = fp_type

    def create(self, snip=None):
        """
        Creates quad hashes for a given audio file
        """
        q, r, c, w, h = self.params
        # print("pegou params")
        samples = load_audio(self.path, snip=snip)
        # print("carregou audio")
        spectrogram = stft(samples)
        # print("fez o spectograma")
        self.peaks = list(find_peaks(spectrogram, w, h))
        # self.save_spectrogram_with_peaks(spectrogram, self.peaks)
        # print("encontrou picos")
        quads = find_quads_seg_tree(self.peaks, r, c)
        # print("encontrou quads")
        self.strongest = n_strongest(spectrogram, quads, q)
        # print("selecionou os mais fortes")
        self.hashes = [generate_hash(q) for q in self.strongest]
        # print("gerou hashes")
    
    def save_spectrogram_with_peaks(self, spectrogram, peaks, out_dir="plots"):
        """
        Salva o espectrograma em escala logarítmica com os picos encontrados.
        
        Args:
            spectrogram (ndarray): matriz do espectrograma
            peaks (list): lista de picos detectados
            out_dir (str): diretório para salvar os plots
        """
        # cria diretório se não existir
        if not os.path.exists(out_dir):
            os.makedirs(out_dir)

        audio_filename = os.path.splitext(os.path.basename(self.path))[0]

        # --- Versão limpa ---
        plt.figure(figsize=(12, 6))
        plt.imshow(
            np.transpose(spectrogram),
            origin="lower",
            aspect="auto",
            cmap="magma"
        )
        plt.colorbar(label="Amplitude (dB)")
        plt.xlabel("Tempo (frames STFT)")
        plt.ylabel("Frequência (bins)")
        plt.title("Spectrograma")
        plt.tight_layout()

        out_path_clean = os.path.join(out_dir, f"{audio_filename}_spectrogram.png")
        plt.savefig(out_path_clean, dpi=150)
        plt.close()
        print(f"Spectrograma (sem picos) salvo em: {out_path_clean}")

        # --- Versão com picos ---
        plt.figure(figsize=(12, 6))
        plt.imshow(
            np.transpose(spectrogram),
            origin="lower",
            aspect="auto",
            cmap="magma"
        )
        x_vals = [p.x for p in peaks]
        y_vals = [p.y for p in peaks]
        plt.scatter(x_vals, y_vals, c="cyan", s=10, marker="x", label="Peaks")
        plt.colorbar(label="Amplitude (dB)")
        plt.xlabel("Tempo (frames STFT)")
        plt.ylabel("Frequência (bins)")
        plt.title("Spectrograma com picos detectados")
        plt.legend()
        plt.tight_layout()

        out_path_peaks = os.path.join(out_dir, f"{audio_filename}_spectrogram_peaks.png")
        plt.savefig(out_path_peaks, dpi=150)
        plt.close()
        print(f"Spectrograma (com picos) salvo em: {out_path_peaks}")




class ReferenceFingerprint(Fingerprint):

    def __init__(self, path):
        self.fp_type = fpType.Reference
        Fingerprint.__init__(self, path, fp_type=self.fp_type)

    def create(self, pickle_dir="fingerprints"):
        """
        Creates fingerprints and saves object attributes to pickle file
        
        Args:
            pickle_dir (str): Directory to save pickle files (default: "fingerprints")
        """
        # Create the fingerprints using parent method
        Fingerprint.create(self)
        
        # Save to pickle file
        self.save_to_pickle(pickle_dir)
    
    def save_to_pickle(self, pickle_dir="fingerprints"):
        """
        Saves the fingerprint attributes to a pickle file
        
        Args:
            pickle_dir (str): Directory to save pickle files
        """
        # Create directory if it doesn't exist
        if not os.path.exists(pickle_dir):
            os.makedirs(pickle_dir)
        
        # Generate filename from audio file path
        audio_filename = os.path.splitext(os.path.basename(self.path))[0]
        pickle_filename = f"{audio_filename}_fingerprint.pkl"
        pickle_path = os.path.join(pickle_dir, pickle_filename)
        
        # Convert namedtuples to simple tuples to avoid pickle issues
        peaks_data = [(peak.x, peak.y) for peak in self.peaks]
        
        # Convert strongest quads to serializable format
        strongest_data = []
        for quad in self.strongest:
            quad_data = {
                'A': (quad.A.x, quad.A.y),
                'C': (quad.C.x, quad.C.y),
                'D': (quad.D.x, quad.D.y),
                'B': (quad.B.x, quad.B.y)
            }
            strongest_data.append(quad_data)
        
        # Prepare data to pickle (all important attributes)
        fingerprint_data = {
            'path': self.path,
            'fp_type': self.fp_type,
            'params': self.params,
            'peaks': peaks_data,
            'strongest': strongest_data,
            'hashes': self.hashes
        }
        
        # Save to pickle file
        with open(pickle_path, 'wb') as f:
            pickle.dump(fingerprint_data, f)
        
        print(f"Fingerprint saved to: {pickle_path}")
    
    @classmethod
    def load_from_pickle(cls, pickle_path):
        """
        Recreates a ReferenceFingerprint object from a pickle file
        
        Args:
            pickle_path (str): Path to the pickle file
            
        Returns:
            ReferenceFingerprint: Recreated fingerprint object
        """
        from collections import namedtuple
        
        with open(pickle_path, 'rb') as f:
            fingerprint_data = pickle.load(f)
        
        # Create new instance
        fingerprint = cls(fingerprint_data['path'])
        
        # Restore basic attributes
        fingerprint.fp_type = fingerprint_data['fp_type']
        fingerprint.params = fingerprint_data['params']
        fingerprint.hashes = fingerprint_data['hashes']
        
        # Recreate namedtuples
        Peak = namedtuple('Peak', ['x', 'y'])
        Quad = namedtuple('Quad', ['A', 'C', 'D', 'B'])
        
        # Restore peaks from tuple data
        fingerprint.peaks = [Peak(x, y) for x, y in fingerprint_data['peaks']]
        
        # Restore strongest quads from dict data
        fingerprint.strongest = []
        for quad_data in fingerprint_data['strongest']:
            quad = Quad(
                A=Peak(quad_data['A'][0], quad_data['A'][1]),
                C=Peak(quad_data['C'][0], quad_data['C'][1]),
                D=Peak(quad_data['D'][0], quad_data['D'][1]),
                B=Peak(quad_data['B'][0], quad_data['B'][1])
            )
            fingerprint.strongest.append(quad)
        
        return fingerprint


class QueryFingerprint(Fingerprint):

    def __init__(self, path):
        self.fp_type = fpType.Query
        Fingerprint.__init__(self, path, fp_type=self.fp_type)

    def create(self):
        Fingerprint.create(self, snip=15)


In [15]:
from __future__ import division, print_function
from collections import defaultdict, namedtuple
import numpy as np
import os
import math
import operator
import faiss  # pip install faiss-cpu (ou faiss-gpu)

try:
    from itertools import izip
except ImportError:
    izip = zip
    xrange = range


class InMemoryQfpDB:
    """
    Implementação em memória das estruturas:
     - fidindex -> dict (title -> metadata)
     - peakfile -> numpy arrays concatenados (peaks_x, peaks_y) com offsets por record
     - refrecords -> quads armazenados em numpy arrays (Ax,Ay,Cx,Cy,Dx,Dy,Bx,By) + recordid
     - searchtree -> FAISS index sobre hashes (4-dim float32)
    """

    def __init__(self, faiss_metric='L2'):
        # metadata index: title -> dict {recordid, num_peaks, peak_start, peak_end, num_quads, quad_start, quad_end}
        self.fidindex = {}
        # counters
        self._next_recordid = 1

        # peakfile: 1D arrays for X (time) and Y (freq) (dtype=int32)
        self.peaks_x = np.empty((0,), dtype=np.int32)
        self.peaks_y = np.empty((0,), dtype=np.int32)
        self.peak_offsets = {}  # recordid -> (start, end)

        # refrecords / quads: store quad coordinates as int32 columns
        self.quad_Ax = np.empty((0,), dtype=np.int32)
        self.quad_Ay = np.empty((0,), dtype=np.int32)
        self.quad_Cx = np.empty((0,), dtype=np.int32)
        self.quad_Cy = np.empty((0,), dtype=np.int32)
        self.quad_Dx = np.empty((0,), dtype=np.int32)
        self.quad_Dy = np.empty((0,), dtype=np.int32)
        self.quad_Bx = np.empty((0,), dtype=np.int32)
        self.quad_By = np.empty((0,), dtype=np.int32)
        self.quad_recordid = np.empty((0,), dtype=np.int32)

        # hashes array (float32 Nx4) in same order as quads
        self.hashes = np.empty((0, 4), dtype=np.float32)

        # FAISS index (will be created lazily). We keep index_flat for simplicity.
        self.faiss_index = None
        self.faiss_ids_offset = 0  # corresponds 1-to-1 with row indices in self.hashes

        # namedtuples (compat)
        self.Peak = namedtuple('Peak', ['x', 'y'])
        self.Quad = namedtuple('Quad', ['A', 'C', 'D', 'B'])
        mcNames = ['recordid', 'offset', 'num_matches', 'sTime', 'sFreq']
        self.MatchCandidate = namedtuple('MatchCandidate', mcNames)
        self.Match = namedtuple('Match', ['record', 'offset', 'vScore'])

    # --------------------
    # Utilities: internal
    # --------------------

    def _ensure_faiss_index(self, use_gpu=False):
        """
        Builds a FAISS IndexFlatL2 over self.hashes if not exists.
        For large DBs prefer IVF,PQ or HNSW depending on memory/speed tradeoffs.
        """
        if self.faiss_index is not None:
            return
        d = 4
        # index that stores vectors and supports range_search
        index = faiss.IndexFlatL2(d)
        # convert to float32 contiguous
        if self.hashes.shape[0] > 0:
            index.add(self.hashes)
        self.faiss_index = index

    # --------------------
    # STORING FUNCTIONS
    # --------------------

    def store(self, fp, title):
        """
        Store a ReferenceFingerprint (in memory).
        """
        if fp.fp_type != fpType.Reference:
            raise TypeError("May only store reference fingerprints in db")

        if title in self.fidindex:
            print(f"record already exists: {title}")
            return

        recordid = self._next_recordid
        self._next_recordid += 1

        # 1) store peaks: fp.peaks is list of namedtuples Peak(x,y)
        peaks_arr_x = np.array([int(p.x) for p in fp.peaks], dtype=np.int32)
        peaks_arr_y = np.array([int(p.y) for p in fp.peaks], dtype=np.int32)
        start = len(self.peaks_x)
        self.peaks_x = np.concatenate([self.peaks_x, peaks_arr_x]) if peaks_arr_x.size > 0 else self.peaks_x
        self.peaks_y = np.concatenate([self.peaks_y, peaks_arr_y]) if peaks_arr_y.size > 0 else self.peaks_y
        end = len(self.peaks_x)
        self.peak_offsets[recordid] = (start, end)

        # 2) store quads (strongest) and hashes
        # Each quad is namedtuple with A,C,D,B where each .x,.y may be numpy scalars -> cast to int
        n_quads = len(fp.strongest)
        if n_quads > 0:
            Ax = np.array([int(q.A.x) for q in fp.strongest], dtype=np.int32)
            Ay = np.array([int(q.A.y) for q in fp.strongest], dtype=np.int32)
            Cx = np.array([int(q.C.x) for q in fp.strongest], dtype=np.int32)
            Cy = np.array([int(q.C.y) for q in fp.strongest], dtype=np.int32)
            Dx = np.array([int(q.D.x) for q in fp.strongest], dtype=np.int32)
            Dy = np.array([int(q.D.y) for q in fp.strongest], dtype=np.int32)
            Bx = np.array([int(q.B.x) for q in fp.strongest], dtype=np.int32)
            By = np.array([int(q.B.y) for q in fp.strongest], dtype=np.int32)
            recids = np.full((n_quads,), recordid, dtype=np.int32)

            self.quad_Ax = np.concatenate([self.quad_Ax, Ax]) if Ax.size > 0 else self.quad_Ax
            self.quad_Ay = np.concatenate([self.quad_Ay, Ay]) if Ay.size > 0 else self.quad_Ay
            self.quad_Cx = np.concatenate([self.quad_Cx, Cx]) if Cx.size > 0 else self.quad_Cx
            self.quad_Cy = np.concatenate([self.quad_Cy, Cy]) if Cy.size > 0 else self.quad_Cy
            self.quad_Dx = np.concatenate([self.quad_Dx, Dx]) if Dx.size > 0 else self.quad_Dx
            self.quad_Dy = np.concatenate([self.quad_Dy, Dy]) if Dy.size > 0 else self.quad_Dy
            self.quad_Bx = np.concatenate([self.quad_Bx, Bx]) if Bx.size > 0 else self.quad_Bx
            self.quad_By = np.concatenate([self.quad_By, By]) if By.size > 0 else self.quad_By
            self.quad_recordid = np.concatenate([self.quad_recordid, recids]) if recids.size > 0 else self.quad_recordid

        # 3) store hashes (fp.hashes assumed iterable of 4-tuples or np.array)
        if len(fp.hashes) > 0:
            h = np.array(fp.hashes, dtype=np.float32).reshape(-1, 4)
            self.hashes = np.vstack([self.hashes, h]) if self.hashes.size else h

        # 4) update fidindex metadata
        self.fidindex[title] = {
            'recordid': recordid,
            'title': title,
            'num_peaks': len(fp.peaks),
            'peak_start': start,
            'peak_end': end,
            'num_quads': n_quads,
            'quad_start': len(self.hashes) - n_quads if n_quads > 0 else 0,
            'quad_end': len(self.hashes)
        }

        # 5) invalidate/rebuild faiss (lazy)
        # simplest approach: discard index; build on next query
        if self.faiss_index is not None:
            self.faiss_index = None

        print(f"Stored record '{title}' with recordid {recordid}, peaks {len(fp.peaks)}, quads {n_quads}")

    def store_from_pickle(self, pickle_path, title=None):
        fp = ReferenceFingerprint.load_from_pickle(pickle_path)
        if title is None:
            title = os.path.splitext(os.path.basename(fp.path))[0]
        self.store(fp, title)

    def store_all_pickles_from_directory(self, pickle_dir="fingerprints"):
        if not os.path.exists(pickle_dir):
            print(f"Directory {pickle_dir} does not exist")
            return
        pickle_files = [f for f in os.listdir(pickle_dir) if f.endswith('.pkl') or f.endswith('.pkl') or f.endswith('.pickle') or f.endswith('.pkl')]
        if not pickle_files:
            print(f"No pickle files found in {pickle_dir}")
            return
        print(f"Found {len(pickle_files)} pickle files to process...")
        for pickle_file in pickle_files:
            try:
                self.store_from_pickle(os.path.join(pickle_dir, pickle_file))
            except Exception as e:
                print(f"Error processing {pickle_file}: {e}")

    # --------------------
    # QUERY / SEARCH FLOW
    # --------------------

    def query(self, fp, vThreshold=0.5, e=0.02, radius_l2=None):
        """
        Query the in-memory DB with a QueryFingerprint.

        Args:
            fp: QueryFingerprint (fp.fp_type must be fpType.Query)
            vThreshold: validation threshold for vScore
            e: per-dimension tolerance used in original implementation
            radius_l2: override radius (Euclidean). If None, computed from e as L_inf->L2: 2*e
        """
        if fp.fp_type != fpType.Query:
            raise TypeError("May only query db with query fingerprints")

        # prepare query peaks sorted by x for verification stage
        qPeaks = [(int(p.x), int(p.y)) for p in fp.peaks]
        qPeaks.sort(key=lambda p: p[0])  # sort by time X
        fp._qPeaks_sorted = qPeaks  # attach for use in verify

        # ensure faiss index
        self._ensure_faiss_index()

        if radius_l2 is None:
            # map L_inf epsilon to L2 radius (4 dims): L2_radius = sqrt(d) * e  (worst-case)
            radius = math.sqrt(4) * e
        else:
            radius = float(radius_l2)

        # For each query hash -> range search
        filtered = defaultdict(list)  # recordid -> list of (offset, (sTime,sFreq))
        for qHash, qQuad in zip(fp.hashes, fp.strongest):
            qvec = np.array(qHash, dtype=np.float32).reshape(1, -1)
            # faiss range_search expects squared radius (L2)
            lims, D, I = self.faiss_index.range_search(qvec, radius * radius)
            # I contains indices of neighbors; D squared L2 distances; lims delimit results
            # print("candidates: ", I)
            if I.size == 0:
                continue
            # iterate matched indices
            for idx in I:
                # retrieve quad coords and recordid
                cQuad, recordid = self._lookup_quad_by_index(int(idx))
                # Now perform identical checks as original _filter_candidates
                # safe-cast to float to avoid int division
                try:
                    # rough pitch coherence:
                    if not 1 / (1 + e) <= (float(qQuad.A.y) / float(cQuad.A.y)) <= 1 / (1 - e):
                        continue
                    # sTime
                    denom = (cQuad.B.x - cQuad.A.x)
                    if denom == 0:
                        continue
                    sTime = (qQuad.B.x - qQuad.A.x) / denom
                    if not 1 / (1 + e) <= sTime <= 1 / (1 - e):
                        continue
                    denom2 = (cQuad.B.y - cQuad.A.y)
                    if denom2 == 0:
                        continue
                    sFreq = (qQuad.B.y - qQuad.A.y) / denom2
                    if not 1 / (1 + e) <= sFreq <= 1 / (1 - e):
                        continue
                    # fine pitch coherence
                    if not abs(qQuad.A.y - (cQuad.A.y * sFreq)) <= 1.8:
                        continue
                    offset = cQuad.A.x - (qQuad.A.x / sTime)
                    print("match recordid: ", recordid)
                    filtered[recordid].append((offset, (sTime, sFreq)))
                except Exception:
                    # numeric issues -> skip candidate
                    continue
        print("filtered candidates: ", filtered)
        # bin times and produce match candidates
        binned = {k: self._bin_times(v) for k, v in filtered.items()}
        results = {k: self._scales(v) for k, v in binned.items() if len(v) >= 4}
        mc = [self.MatchCandidate(k, a[0], a[1], a[2][0], a[2][1])
              for k, v in results.items() for a in v]
        print("match candidates: ", mc)
        
        # validate each candidate
        matches = []
        for m in mc:
            vScore = self._validate_match(m, fp)
            if vScore >= vThreshold:
                title = self._lookup_record_title(m.recordid)
                matches.append(self.Match(title, m.offset, vScore))
        fp.match_candidates = mc
        fp.matches = matches
        return fp.matches

    # --------------------
    # Helper methods mirroring original behaviour
    # --------------------

    def _lookup_quad_by_index(self, idx):
        """
        Given a row index in self.hashes / quads, returns Quad(namedtuple) and recordid
        """
        A = self.Peak(int(self.quad_Ax[idx]), int(self.quad_Ay[idx]))
        C = self.Peak(int(self.quad_Cx[idx]), int(self.quad_Cy[idx]))
        D = self.Peak(int(self.quad_Dx[idx]), int(self.quad_Dy[idx]))
        B = self.Peak(int(self.quad_Bx[idx]), int(self.quad_By[idx]))
        recordid = int(self.quad_recordid[idx])
        return self.Quad(A, C, D, B), recordid

    def _bin_times(self, l, binwidth=3, ts=4):
        d = defaultdict(list)
        for offset, (sTime, sFreq) in l:
            # Escalar offset pelo fator de tempo (artigo do QFP)
            scaled_offset = offset * sTime
            # Colocar no bin mais próximo
            binname = int(math.floor(scaled_offset / binwidth) * binwidth)
            d[binname].append((sTime, sFreq))

        # filtrar bins com pelo menos ts elementos
        return {k: v for k, v in d.items() if len(v) >= ts}


    def _outlier_removal(self, d):
        means = np.mean(d, axis=0)
        stds = np.std(d, axis=0)
        d = [v for v in d if
             (means[0] - 2 * stds[0] <= v[0] <= means[0] + 2 * stds[0]) and
             (means[1] - 2 * stds[1] <= v[1] <= means[1] + 2 * stds[1])]
        return d

    def _scales(self, d):
        o_rm = {k: self._outlier_removal(v) for k, v in d.items()}
        res = [(i[0], len(i[1]), np.mean(i[1], axis=0))
               for i in o_rm.items() if len(i[1]) >= 4]
        sorted_mc = sorted(res, key=operator.itemgetter(1), reverse=True)
        return sorted_mc

    def _lookup_peak_range(self, recordid, offset, e=3750):
        """
        Return peaks for given recordid in range [offset, offset+e]
        """
        if recordid not in self.peak_offsets:
            return []
        start, end = self.peak_offsets[recordid]
        xs = self.peaks_x[start:end]
        ys = self.peaks_y[start:end]
        # filter by X range
        mask = (xs >= offset) & (xs <= offset + e)
        sel_x = xs[mask]
        sel_y = ys[mask]
        return [self.Peak(int(x), int(y)) for x, y in zip(sel_x, sel_y)]

    def _verify_peaks(self, mc, rPeaks, qPeaks, eX=18, eY=12):
        """
        very similar to original: use bisect on qPeaks (sorted by x)
        qPeaks: list of tuples (x,y)
        rPeaks: list of Peak namedtuples (x,y)
        """
        validated = 0
        if len(rPeaks) == 0:
            return 0.0
        # qPeaks sorted by x
        xs = [p[0] for p in qPeaks]
        from bisect import bisect_left, bisect_right
        for rPeak in rPeaks:
            rPeak_adj_x = rPeak.x - mc.offset
            rPeakScaled_x = rPeak_adj_x / mc.sFreq
            rPeakScaled_y = rPeak.y / mc.sTime
            lBound = bisect_left(xs, (rPeakScaled_x - eX))
            rBound = bisect_right(xs, (rPeakScaled_x + eX))
            for i in range(lBound, rBound):
                if not (rPeakScaled_y - eY <= qPeaks[i][1] <= rPeakScaled_y + eY):
                    continue
                else:
                    validated += 1
        vScore = (float(validated) / len(rPeaks))
        return vScore

    def _validate_match(self, mc, fp):
        """
        mc is MatchCandidate(recordid, offset, num_matches, sTime, sFreq)
        """
        rPeaks = self._lookup_peak_range(mc.recordid, mc.offset)
        # Ensure qPeaks sorted stored in fp._qPeaks_sorted
        qPeaks = fp._qPeaks_sorted
        # create a simple object with fields used by _verify_peaks: mc has recordid, offset, sTime, sFreq
        # For compatibility, create a small namedtuple
        M = namedtuple('M', ['recordid', 'offset', 'sTime', 'sFreq'])
        mm = M(mc.recordid, mc.offset, mc.sTime, mc.sFreq)
        vScore = self._verify_peaks(mm, rPeaks, qPeaks)
        title = self._lookup_record_title(mc.recordid)
        return vScore

    def _lookup_record_title(self, recordid):
        # reverse lookup in fidindex
        for title, meta in self.fidindex.items():
            if meta['recordid'] == recordid:
                return title
        return None


In [16]:
# filenames = glob.glob("/home/luiz/repositories/qfp/ecad_db/BAF/audio/references/*.wav")

# start = time.time()

# for file in filenames:
#     filename = os.path.splitext(os.path.basename(file))[0]
#     print(filename)
#     fp_r = ReferenceFingerprint(file)
#     fp_r.create("/home/luiz/repositories/qfp_original2/qfp/data/baf_references")

# end = time.time() 
# print(f"Tempo de execução: {end - start:.4f} segundos")

In [17]:
# db = InMemoryQfpDB()

# # armazenar um fingerprint salvo em pickle
# # db.store_from_pickle("fingerprints/song1_fingerprint.pkl")

# # armazenar vários
# db.store_all_pickles_from_directory("/home/luiz/repositories/qfp_original2/qfp/scripts/data/cutted_cia_av_references")


In [18]:
# qfp = QueryFingerprint("/home/luiz/repositories/qfp_original2/qfp/data/cutted/queries/12087188.ogg")
# qfp.create()  # gera hashes, peaks, strongest

In [19]:
# matches = db.query(qfp, vThreshold=0.125)
# print("Matches:", matches)

In [20]:
from bisect import bisect_left, bisect_right
from collections import namedtuple
from itertools import combinations


def find_quads(peaks, r, c):
    """
    Returns list of valid/strong quads for list of peaks
    """
    quads = []
    for root in peaks:
        quads += _root_quads(root, peaks, r, c)
    return quads


def _root_quads(root, peaks, r, c):
    """
    finds valid quads for given root
    """
    quads = []
    filtered = _filter_peaks(root, peaks, r, c)
    if filtered is None:
        return []
    found = _valid_quads(root, filtered)
    if found is not None:
        quads += found
    return quads


def _filter_peaks(root, peaks, r, c):
    """
    returns peaks inside window of Ax + c ± (r / 2)
    """
    lastPeak = peaks[-1].x
    windowStart = root.x + c - (r / 2)
    if windowStart > lastPeak:
        return None
    windowEnd = windowStart + r
    idx_start = bisect_left(peaks, (windowStart, 0))
    idx_end = bisect_right(peaks, (windowEnd, 0))
    filtered = peaks[idx_start:idx_end]
    if len(filtered) < 3:
        return None
    return filtered


def _valid_quads(root, filtered):
    """
    returns list of validated quads for given root (A)
    """
    Quad = namedtuple('Quad', ['A', 'C', 'D', 'B'])
    validQuads = []
    for comb in combinations(filtered, 3):
        quad = Quad(root, comb[0], comb[1], comb[2])
        if _valid_quad(quad):
            validQuads.append(quad)
    if len(validQuads) == 0:
        return None
    else:
        return validQuads


def _valid_quad(q):
    """
    Evaluates:

          Ay < By
      Ax < Cx <= Dx <= Bx
      Ay < Cy ,  Dy <= By

    !! NOTE: assumes combinations are sorted by x value
    (default behavior of itertools.combinations)
    """
    if q.A.y < q.C.y < q.B.y and q.A.y < q.D.y <= q.B.y:
        return True
    else:
        return False


In [21]:
from bisect import bisect_left, bisect_right
from heapq import heappush, heappushpop
from collections import namedtuple

Quad = namedtuple('Quad', ['A', 'C', 'D', 'B'])
def find_quads_stream_v2(peaks, r, c, spec, q_per_partition, partition_len=250):
    """
    Implementação completa e eficiente.
    """
    from bisect import bisect_left, bisect_right
    from heapq import heappush, heappushpop

    heaps = {}
    counter = 0
    lastPeakX = peaks[-1].x if peaks else 0

    for root in peaks:
        windowStart = root.x + c - (r / 2)
        if windowStart > lastPeakX:
            continue
        windowEnd = windowStart + r
        idx_start = bisect_left(peaks, (windowStart, 0))
        idx_end = bisect_right(peaks, (windowEnd, 0))
        filtered = peaks[idx_start:idx_end]
        if len(filtered) < 3:
            continue

        # Precompute arrays for filtered to speed up scans
        xs = [p.x for p in filtered]
        ys = [p.y for p in filtered]
        # find leftmost index where x > root.x (Ax < Cx)
        # if no such index, continue
        import bisect
        left_idx = bisect.bisect_right(xs, root.x - 1)  # first with x > root.x
        if left_idx >= len(filtered):
            continue

        # We'll iterate B over indices b_idx in [left_idx .. len(filtered)-1]
        for b_idx in range(left_idx, len(filtered)):
            B = filtered[b_idx]
            # By must be > Ay (root.y)
            if not (B.y > root.y):
                continue
            # For this B, valid C and D must satisfy:
            # - Ax < Cx <= Bx, Ay < Cy < By
            # - Ax < Dx <= Bx, Ay < Dy <= By
            # So consider candidates between left_idx and b_idx inclusive
            # Build list of indices in [left_idx .. b_idx] that satisfy y constraints for C and D
            # Because y constraints differ slightly between C and D, we can compute two index lists
            c_indices = []
            d_indices = []
            for i in range(left_idx, b_idx + 1):
                yi = ys[i]
                if yi > root.y and yi < B.y:
                    c_indices.append(i)
                if yi > root.y and yi <= B.y:
                    d_indices.append(i)
            if not c_indices or not d_indices:
                continue

            # Now produce pairs (C_idx, D_idx) with C_idx <= D_idx (i.e., x(C) <= x(D))
            # c_indices and d_indices are in ascending x order, so we can produce pairs efficiently:
            # for each c_idx, we can find in d_indices the first position >= c_idx
            # use bisect on d_indices (which contains indices in original filtered space)
            import bisect as _bis
            # create an array for binary-searchable d_indices
            # iterate over c_indices
            for c_idx in c_indices:
                # find first d_pos in d_indices where value >= c_idx
                pos = _bis.bisect_left(d_indices, c_idx)
                if pos >= len(d_indices):
                    continue
                for d_pos in range(pos, len(d_indices)):
                    d_idx = d_indices[d_pos]
                    C = filtered[c_idx]
                    D = filtered[d_idx]
                    # extra safety check: C.x <= D.x <= B.x (holds because indices in range and xs sorted)
                    # Build quad
                    quad = Quad(root, C, D, B)
                    # compute strength using spec (C.x, C.y) and (D.x, D.y)
                    try:
                        strength = spec[C.x][C.y] + spec[D.x][D.y]
                    except Exception:
                        # Caso índice inválido por qualquer razão — pular
                        continue

                    # push into heap for partition
                    part_idx = root.x // partition_len
                    heap = heaps.get(part_idx)
                    entry = (strength, counter, quad)
                    counter += 1
                    if heap is None:
                        # initialize with capacity q_per_partition
                        heaps[part_idx] = [entry]
                    else:
                        # maintain min-heap of up to q_per_partition strongest; smallest strength at root
                        # we want to keep largest strengths; use heappushpop to maintain size
                        if len(heap) < q_per_partition:
                            import heapq
                            heapq.heappush(heap, entry)
                        else:
                            import heapq
                            # if entry stronger than smallest, replace
                            if entry[0] > heap[0][0]:
                                heapq.heapreplace(heap, entry)

    # flatten heaps in order of partition index -> return list of quads (strongest per partition)
    result = []
    for pidx in sorted(heaps.keys()):
        # heaps[pidx] is min-heap; extract entries and sort descending by strength for determinism
        heap = heaps[pidx]
        entries = sorted(heap, key=lambda e: (-e[0], e[1]))
        result.extend([e[2] for e in entries])
    return result


In [22]:
# filenames = glob.glob("/home/luiz/repositories/qfp/ecad_db/BAF/audio/references/*.wav")

# start = time.time()

# for file in filenames:
#     filename = os.path.splitext(os.path.basename(file))[0]
#     print(filename)
#     fp_r = ReferenceFingerprint(file)
#     fp_r.create("/home/luiz/repositories/qfp_original2/qfp/data/baf_references")

# end = time.time() 
# print(f"Tempo de execução: {end - start:.4f} segundos")

In [29]:
def compare_methods(audio_path):
    """
    Compara quads gerados por:
      1) find_quads + n_strongest
      2) find_quads_stream_v2
    """
    print(f"=== Testando com áudio: {audio_path} ===")

    # fingerprint com params Reference (usa poucos quads)
    fp = Fingerprint(audio_path, fpType.Reference)

    # # carrega áudio e spectrograma
    q, r, c, w, h = fp.params
    # samples = fpType  # só para referência
    # samples = fp.create()  # mas esse já roda find_quads original...
    # então vamos repetir manualmente só a parte que interessa

    samples = load_audio(audio_path)
    spectrogram = stft(samples)
    peaks = list(find_peaks(spectrogram, w, h))

    # --- Método 1: find_quads + n_strongest ---
    quads_all = list(find_quads(peaks, r, c))
    strongest_1 = n_strongest(spectrogram, quads_all, q)
    # hashes_1 = set(generate_hash(qd) for qd in strongest_1)
    print(f"Método 1: {len(quads_all)} quads totais, {len(strongest_1)} mais fortes")

    # --- Método 2: find_quads_stream_v2 ---
    strongest_2 = find_quads_stream_v2(peaks, r, c, spectrogram, q)
    # hashes_2 = set(generate_hash(qd) for qd in strongest_2)
    print(f"Método 2: {len(strongest_2)} mais fortes (streaming)")

    print("Quads metodo 1: ", strongest_1)
    print("Quads metodo 2: ", strongest_2)

    set1 = set(strongest_1)
    set2 = set(strongest_2)

    print("Resultados iguais? ", set1 == set2)
    # # --- Comparação ---
    # diff1 = hashes_1 - hashes_2
    # diff2 = hashes_2 - hashes_1

    # if not diff1 and not diff2:
    #     print("✅ Os conjuntos de hashes coincidem!")
    # else:
    #     print("⚠️ Diferença encontrada!")
    #     print(f"Presentes só no método 1: {len(diff1)}")
    #     print(f"Presentes só no método 2: {len(diff2)}")




In [30]:
audio_file = "/home/luiz/repositories/qfp_original2/qfp/data/cutted/references/1160248.ogg"
compare_methods(audio_file)

=== Testando com áudio: /home/luiz/repositories/qfp_original2/qfp/data/cutted/references/1160248.ogg ===


Método 1: 572 quads totais, 135 mais fortes
Método 2: 171 mais fortes (streaming)
Quads metodo 1:  [Quad(A=Peak(x=np.int64(141), y=np.int64(14)), C=Peak(x=np.int64(468), y=np.int64(18)), D=Peak(x=np.int64(501), y=np.int64(127)), B=Peak(x=np.int64(501), y=np.int64(169))), Quad(A=Peak(x=np.int64(141), y=np.int64(14)), C=Peak(x=np.int64(468), y=np.int64(18)), D=Peak(x=np.int64(501), y=np.int64(127)), B=Peak(x=np.int64(533), y=np.int64(498))), Quad(A=Peak(x=np.int64(141), y=np.int64(14)), C=Peak(x=np.int64(468), y=np.int64(18)), D=Peak(x=np.int64(501), y=np.int64(127)), B=Peak(x=np.int64(549), y=np.int64(334))), Quad(A=Peak(x=np.int64(141), y=np.int64(14)), C=Peak(x=np.int64(468), y=np.int64(18)), D=Peak(x=np.int64(501), y=np.int64(127)), B=Peak(x=np.int64(559), y=np.int64(295))), Quad(A=Peak(x=np.int64(141), y=np.int64(14)), C=Peak(x=np.int64(468), y=np.int64(18)), D=Peak(x=np.int64(501), y=np.int64(127)), B=Peak(x=np.int64(563), y=np.int64(388))), Quad(A=Peak(x=np.int64(141), y=np.int64(